# Merge CAFA5 & CAFA6 Train Datasets

In [1]:
import pandas as pd
import numpy as np

In [2]:
cafa5_train_terms = pd.read_csv('../data/cafa-5-protein-function-prediction/Train/train_terms.tsv', sep='\t')
cafa6_train_terms = pd.read_csv('../data/cafa-6-protein-function-prediction/Train/train_terms.tsv', sep='\t')

In [12]:
print(len(cafa5_train_terms))
print(len(cafa6_train_terms))

print(cafa5_train_terms.aspect.unique())
print(cafa6_train_terms.aspect.unique())

print(len(cafa5_train_terms.EntryID.unique()))
print(len(cafa6_train_terms.EntryID.unique()))

5363863
537027
['BPO' 'CCO' 'MFO']
['CCO' 'MFO' 'BPO']
142246
82404


In [4]:
# Standardize the aspect column

ASPECT_MAP = {"P": "BPO", "C": "CCO", "F": "MFO"}

cafa6_train_terms["aspect"] = cafa6_train_terms["aspect"].map(ASPECT_MAP)

In [5]:
# Merge annotations: stack rows, then dedupe (EntryID, term).

cafa5_cafa6_train_terms = (
    pd.concat([cafa5_train_terms, cafa6_train_terms], ignore_index=True)
    .drop_duplicates(subset=["EntryID", "term"])
)

print(f"Merged rows: {len(cafa5_cafa6_train_terms):,}")
print(f"Unique proteins: {cafa5_cafa6_train_terms['EntryID'].nunique():,}")
print(f"RAM ~{cafa5_cafa6_train_terms.memory_usage(deep=True).sum() / 1e6:.0f} MB")

Merged rows: 5,410,821
Unique proteins: 145,382
RAM ~943 MB


In [6]:
print(len(cafa5_cafa6_train_terms))

print(cafa5_cafa6_train_terms.aspect.unique())

5410821
['BPO' 'CCO' 'MFO']


In [7]:
# Check missing values and duplicates
print(cafa5_cafa6_train_terms.isnull().sum())
print(cafa5_cafa6_train_terms.duplicated().sum())


EntryID    0
term       0
aspect     0
dtype: int64
0


## Merge FASTA 

Union ~145K proteins without loading both full FASTA files into a dict. Writes `data/cafa-5-cafa-6-protein-function-prediction/Train/train_sequences.fasta`

In [8]:
from pathlib import Path

MERGED_DIR = Path("../data/cafa-5-cafa-6-protein-function-prediction/Train")
MERGED_DIR.mkdir(parents=True, exist_ok=True)

CAFA5_FASTA = Path("../data/cafa-5-protein-function-prediction/Train/train_sequences.fasta")
CAFA6_FASTA = Path("../data/cafa-6-protein-function-prediction/Train/train_sequences.fasta")
OUT_FASTA = MERGED_DIR / "train_sequences.fasta"


def extract_entry_id(header: str) -> str:
    h = header.strip().lstrip(">")
    if "|" in h:
        parts = h.split("|")
        if len(parts) >= 2 and parts[1]:
            return parts[1]
    return h.split()[0]


def stream_fasta(path: Path):
    header, chunks = None, []
    with path.open() as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if header is not None:
                    yield extract_entry_id(header), header, "".join(chunks)
                header, chunks = line, []
            elif line:
                chunks.append(line)
        if header is not None:
            yield extract_entry_id(header), header, "".join(chunks)


seen_ids: set[str] = set()
written = 0

with OUT_FASTA.open("w") as out:
    for fasta_path in (CAFA5_FASTA, CAFA6_FASTA):
        for entry_id, header, seq in stream_fasta(fasta_path):
            if entry_id in seen_ids:
                continue
            seen_ids.add(entry_id)
            out.write(f"{header}\n")
            for i in range(0, len(seq), 80):
                out.write(seq[i : i + 80] + "\n")
            written += 1

print(f"Wrote {written:,} sequences → {OUT_FASTA}")
print(f"IDs tracked in memory: {len(seen_ids):,} (~{len(seen_ids) * 72 / 1e6:.0f} MB for ID set)")

Wrote 145,382 sequences → ../data/cafa-5-cafa-6-protein-function-prediction/Train/train_sequences.fasta
IDs tracked in memory: 145,382 (~10 MB for ID set)


In [9]:
# Save merged terms 
OUT_TERMS = MERGED_DIR / "train_terms.tsv"
cafa5_cafa6_train_terms.to_csv(OUT_TERMS, sep="\t", index=False)
print(f"Saved → {OUT_TERMS}")

Saved → ../data/cafa-5-cafa-6-protein-function-prediction/Train/train_terms.tsv
